In [1]:
# [Problem 1] Blending scratch mounting

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Load the dataset
df = pd.read_csv('train.csv') 

# Select features (X) and target (y)
X = df[['GrLivArea', 'YearBuilt']]
y = df['SalePrice']

# Split the data (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")


Training set size: 1168
Validation set size: 292


In [ ]:
#2. Training Single Models


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_val)
lr_mse = mean_squared_error(y_val, lr_pred)
print(f"Linear Regression MSE: {lr_mse:.2f}")

# 2. Support Vector Regressor (with scaling - important for SVR)
# Using a pipeline to standardize the features first
svr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=100000)) # Increased C for better fitting to numerical data
])
svr_pipe.fit(X_train, y_train)
svr_pred = svr_pipe.predict(X_val)
svr_mse = mean_squared_error(y_val, svr_pred)
print(f"SVR (Scaled) MSE: {svr_mse:.2f}")

# 3. Decision Tree Regressor
dt = DecisionTreeRegressor(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_val)
dt_mse = mean_squared_error(y_val, dt_pred)
print(f"Decision Tree MSE: {dt_mse:.2f}")

# Store predictions for blending
preds = {
    'lr': lr_pred,
    'svr': svr_pred,
    'dt': dt_pred
}

Linear Regression MSE: 2495554898.67
SVR (Scaled) MSE: 2415661828.26
Decision Tree MSE: 1844304720.66


In [ ]:
# Blending Scratch Implementation (Problem 1 Solution)

In [5]:
# Blending 1: Simple Averaging
blend_pred_1 = (preds['lr'] + preds['svr'] + preds['dt']) / 3
blend_mse_1 = mean_squared_error(y_val, blend_pred_1)
print(f"Blending 1 (Simple Average) MSE: {blend_mse_1:.2f}")

Blending 1 (Simple Average) MSE: 2009561386.13


In [ ]:
# Blending Example 2: Weighted Averaging (Manual Weights)

In [6]:
# Blending 2: Weighted Averaging (Manual Heuristic)
w_lr, w_svr, w_dt = 0.3, 0.2, 0.5
blend_pred_2 = w_lr * preds['lr'] + w_svr * preds['svr'] + w_dt * preds['dt']
blend_mse_2 = mean_squared_error(y_val, blend_pred_2)
print(f"Blending 2 (Weighted Average: 0.3, 0.2, 0.5) MSE: {blend_mse_2:.2f}")

Blending 2 (Weighted Average: 0.3, 0.2, 0.5) MSE: 1897954249.91


In [ ]:
#Blending Example 3: Stacking-like Blending (Meta-Learner on Validation Set)

In [7]:
# Blending 3: Linear Meta-Learner (Trained Blender)
# 1. Create the meta-feature matrix (X_meta)
X_meta = np.column_stack((preds['lr'], preds['svr'], preds['dt']))

# 2. Train a meta-learner (Linear Regression) on the validation set predictions/targets
meta_learner = LinearRegression()
# X_meta (features: base model predictions), y_val (target: true prices)
meta_learner.fit(X_meta, y_val)

# 3. Predict using the meta-learner on the same meta-features
blend_pred_3 = meta_learner.predict(X_meta)
blend_mse_3 = mean_squared_error(y_val, blend_pred_3)
print(f"Blending 3 (Linear Meta-Learner) MSE: {blend_mse_3:.2f}")

Blending 3 (Linear Meta-Learner) MSE: 1779285949.99


In [ ]:
#[Problem 2] Scratch mounting of bagging

In [8]:
# Assuming X_train, y_train, X_val, y_val are loaded from the previous step
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Parameters ---
n_estimators = 10
train_size = len(X_train)
random_state = 42
np.random.seed(random_state)

# --- Training Single Model for Baseline Comparison ---
dt_single = DecisionTreeRegressor(max_depth=5, random_state=random_state)
dt_single.fit(X_train, y_train)
dt_single_pred = dt_single.predict(X_val)
dt_single_mse = mean_squared_error(y_val, dt_single_pred)
print(f"Single Decision Tree MSE (Baseline): {dt_single_mse:.2f}")

# --- Bagging Scratch Implementation ---
bagging_predictions = []

for i in range(n_estimators):
    # 1. Bootstrap: Create indices by sampling with replacement
    # We use integers from 0 to train_size-1 as indices
    bootstrap_indices = np.random.choice(
        train_size, size=train_size, replace=True
    )
    
    # Select the bootstrap sample
    X_sample = X_train.iloc[bootstrap_indices]
    y_sample = y_train.iloc[bootstrap_indices]
    
    # 2. Train: Fit a new base model
    dt_model = DecisionTreeRegressor(max_depth=5, random_state=random_state + i)
    dt_model.fit(X_sample, y_sample)
    
    # 3. Predict: Get predictions on the validation set
    y_pred = dt_model.predict(X_val)
    bagging_predictions.append(y_pred)

# 4. Average the estimation results (the blending/averaging part)
# Convert the list of arrays into a 2D array and average across the models (axis=0)
bagging_predictions_array = np.array(bagging_predictions)
bagging_final_pred = np.mean(bagging_predictions_array, axis=0)

# Calculate Bagging MSE
bagging_mse = mean_squared_error(y_val, bagging_final_pred)
print(f"Bagging Ensemble (N={n_estimators}) MSE: {bagging_mse:.2f}")

Single Decision Tree MSE (Baseline): 1844304720.66
Bagging Ensemble (N=10) MSE: 1882113197.99


In [9]:
#[Problem 3] Stacking scratch mounting

In [10]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import numpy as np

# --- 0. Define Models and Parameters ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
N_train = len(X_train)

# Base models (Level 0)
# We re-instantiate them to ensure they are untrained
base_models = [
    ('lr', LinearRegression()),
    ('svr', Pipeline([('scaler', StandardScaler()), ('svr', SVR(C=100000))])), # SVR requires scaling
    ('dt', DecisionTreeRegressor(max_depth=5, random_state=42))
]

# Initialize matrix for Level-1 training features (OOF predictions)
# Shape: (N_train samples, 3 base models)
X_meta_train = np.zeros((N_train, len(base_models)))
# Initialize list to store full-training predictions for Level-1 test features
X_meta_val_list = [] # Will store N predictions, one for each model trained on all data

# --- 1. Generate OOF Predictions (X_meta_train) ---
for model_idx, (name, model) in enumerate(base_models):
    oof_predictions = np.zeros(N_train)
    val_predictions = [] # Predictions for X_val from the 5 folds
    
    print(f"Generating OOF predictions for {name}...")
    
    # Iterate through K-folds
    for fold_idx, (train_idx, oof_idx) in enumerate(kf.split(X_train)):
        # Split data for the current fold
        X_fold_train, X_fold_oof = X_train.iloc[train_idx], X_train.iloc[oof_idx]
        y_fold_train, y_fold_oof = y_train.iloc[train_idx], y_train.iloc[oof_idx]
        
        # Train on the fold training data
        model.fit(X_fold_train, y_fold_train)
        
        # Predict on the OOF fold (used for X_meta_train)
        oof_predictions[oof_idx] = model.predict(X_fold_oof)
        
        # Predict on the FINAL validation set (X_val)
        # This will be averaged later to create the final meta-test features
        val_predictions.append(model.predict(X_val))
    
    # Store the OOF predictions as a feature for the meta-learner
    X_meta_train[:, model_idx] = oof_predictions
    
    # Average the K predictions on X_val to get the final Level-1 test feature
    X_meta_val_list.append(np.mean(val_predictions, axis=0))

# Convert list to array
X_meta_val = np.column_stack(X_meta_val_list)

# --- 2. Train the Meta-Learner (Level 1) ---
# The meta-learner is trained on the OOF predictions (X_meta_train) and the true training targets (y_train)
meta_learner = LinearRegression()
meta_learner.fit(X_meta_train, y_train)

# --- 3. Final Prediction on Validation Set (X_val) ---
# The meta-learner predicts on the averaged validation predictions (X_meta_val)
stacking_final_pred = meta_learner.predict(X_meta_val)

# --- 4. Evaluate Stacking Performance ---
stacking_mse = mean_squared_error(y_val, stacking_final_pred)
print("-" * 50)
print(f"Stacking Ensemble MSE: {stacking_mse:.2f}")

# --- Baseline Comparison (Assuming Linear Regression was the best single model) ---
# A typical Linear Regression trained on all X_train
lr_baseline = LinearRegression()
lr_baseline.fit(X_train, y_train)
lr_pred = lr_baseline.predict(X_val)
lr_mse = mean_squared_error(y_val, lr_pred)
print(f"Baseline (Single Linear Regressor) MSE: {lr_mse:.2f}")

# --- 5. Accuracy Check ---
if stacking_mse < lr_mse:
    print(f"\n✅ Stacking is more accurate! (Lower MSE)")
else:
    print(f"\n❌ Stacking was not more accurate in this run.")

Generating OOF predictions for lr...
Generating OOF predictions for svr...
Generating OOF predictions for dt...
--------------------------------------------------
Stacking Ensemble MSE: 2084535945.38
Baseline (Single Linear Regressor) MSE: 2495554898.67

✅ Stacking is more accurate! (Lower MSE)


In [ ]:
#